### Preparing an IDE for Mario 

### Numpy version error

When I started - I faced an from **NumPy 2.x incompatibility** with the older Mario/Gym/NES stack. It’s a known issue with `gym-super-mario-bros`/Mario-style environments and NumPy 2.x. ([GitHub][1])


[1]: https://github.com/Kautenja/gym-super-mario-bros/issues/135?utm_source=chatgpt.com "OverflowError: Python integer 1024 out of bounds for uint8"


### Creating a new environment
At this point, I decided to **create a dedicated `mario` environment**. Cleaner and safer.

You may use Python 3.10 or 3.11, with NumPy below version 2:

```bash
conda create -n mario python=3.10 -y
conda activate mario

python -m pip install "numpy<2"
python -m pip install gym-super-mario-bros nes-py
python -m pip install gymnasium opencv-python matplotlib jupyter ipykernel
```

Then add it to Jupyter:

```bash
python -m ipykernel install --user --name mario --display-name "Python (mario)"
```

Then in Jupyter, choose kernel:

```text
Python (mario)
```

Test:

```python
import numpy as np
print(np.__version__)

import gym_super_mario_bros
env = gym_super_mario_bros.make("SuperMarioBros-1-1-v0")
obs = env.reset()

print("Mario works!")
print(type(obs))
```

I did this to keep my other conda environments clean for ADL and LLM courses and let `mario` be the experimental RL playground.


### Installing CUDA (in a new environment)

**NVIDIA GeForce RTX 2060** is fully capable of running CUDA for deep learning frameworks like PyTorch or TensorFlow.

The secret to making CUDA available in Jupyter is understanding that **Jupyter itself doesn't use the GPU—the Python environment running *inside* Jupyter does**. To make this work smoothly, you should install the CUDA-supported packages into a specific Conda environment and link that environment to Jupyter.

Here is the step-by-step guide to setting it up.

---

## Step 1: Update Your NVIDIA Drivers

Before touching Anaconda, ensure your Windows/Linux NVIDIA drivers are up to date.

* Open **GeForce Experience** and check for updates, or download the latest game-ready/studio driver directly from NVIDIA's website.
* *Note: You do **not** need to install the bulky standalone CUDA Toolkit from NVIDIA on your base system anymore. Conda can handle the toolkit internally.*

## Step 2: Create a Dedicated Conda Environment

Open your **Anaconda Prompt** (or terminal) and create a fresh environment. It's highly recommended to use a specific environment rather than your `base` environment to avoid package conflicts.

```bash
conda create -n cuda_env python=3.10 -y
conda activate cuda_env

```

## Step 3: Install the Deep Learning Framework (with CUDA)

Depending on what you plan to use, install either PyTorch or TensorFlow. Conda will automatically grab the correct compatible CUDA binary files for your RTX 2060.

### Option A: For PyTorch (Recommended)

PyTorch has excellent, modern support via Conda. Run the official command:

```bash
conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia -y

```

### Option B: For TensorFlow

```bash
conda install -c conda-forge cudatoolkit cudnn -y
pip install tensorflow

```

---

## Step 4: Expose the Environment to Jupyter

To make this specific environment show up as an option inside Jupyter Notebook, you need to install a kernel wrapper called `ipykernel`.

While still inside your active `cuda_env`, run:

```bash
conda install -c anaconda ipykernel -y

```

Next, register the environment as a Jupyter kernel:

```bash
python -m ipykernel install --user --name=cuda_env --display-name "Python (RTX 2060 CUDA)"

```

---

## Step 5: Launch and Verify in Jupyter

1. Fire up Jupyter Notebook by running:
```bash
jupyter notebook

```


2. In the top-right corner of Jupyter, click **New** (or click **Kernel** > **Change Kernel** in an open notebook) and select **Python (RTX 2060 CUDA)**.
3. To verify that Jupyter is successfully talking to your RTX 2060, run the relevant code block below in a notebook cell:

### For PyTorch Verification:

```python
import torch
print("Is CUDA available?", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

```

### For TensorFlow Verification:

```python
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

```

> **Quick Troubleshooting Note:** If it returns `False` or `0`, completely shut down Jupyter Notebook, return to the Anaconda Prompt, ensure you are in the `cuda_env`, and launch `jupyter notebook` from there.

``` Bash
nvidia-smi
```

Tue Jun  2 17:38:06 2026
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2060      WDDM  |   00000000:08:00.0  On |                  N/A |
| 25%   36C    P8             21W /  160W |    1175MiB /   6144MiB |      4%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

### ALTERNATIVE: Installing CUDA in mario environment

Because we already have `mario` environment  
and  

RTX 2060 is worth using.

The driver is current enough, and PyTorch wheels include their own CUDA runtime, so we do **not** need to install the full CUDA Toolkit separately. PyTorch recommends using its official install selector for the matching command. ([PyTorch][1])

In `mario`, first uninstall CPU torch:

```bash
pip uninstall torch torchvision torchaudio
```

Then install CUDA PyTorch. Try the official CUDA 12.6 wheel:

```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
```

Then verify:

```bash
python -c "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"
```
#### Output
cuda.get_device_name(0))"  
2.12.0+cu126  
True  
NVIDIA GeForce RTX 2060  

---

It works, now we update `agent.py`:

```python
self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

Then rerun:

```bash
python train.py
```

Small note: the emulator itself still runs on CPU; CUDA helps the CNN training/backprop part.

[1]: https://pytorch.org/get-started/locally/?utm_source=chatgpt.com "Get Started"
